# D9 — mmBERT 도메인 사전학습 2번째 에포크 (1에포크 가중치에서 이어서, Colab)

**무엇**: D8 결과(`models/dapt_mmbert_econ5m_morph`, 검증 손실 1.848 → 0.602)를 출발점으로 같은 형태소 말뭉치를 한 번 더 학습한다.
- 학습률 3e-5(1에포크 5e-5보다 낮게), 다시 6% 예열 후 직선 감소.
- **검증 블록은 1에포크와 같은 2,000개**(`--seed 42`) → 학습 전 손실이 약 0.602로 나오면 제대로 이어받은 것. 학습 순서·가리는 위치는 `--train_seed 43` 으로 바꿔 같은 배치를 반복하지 않음.
- 토큰 캐시는 1에포크 것을 재사용(`--tok_from jhu-clsp/mmBERT-base`) → 토큰화 생략.
- 결과는 새 이름 `dapt_mmbert_econ5m_morph_ep2` 로 저장. **1에포크 가중치는 그대로 남는다.** 어느 쪽을 쓸지는 손실이 아니라 금융 파인튜닝 dev F1로 고른다.

**런타임**: A100 GPU(40GB면 충분, H100 있으면 H100) · RAM 표준. 1에포크 실측 70.5분 → 약 75분, 약 7 CU 예상.
**순서**: 위에서부터 모두 실행. 끊기면 1~4번 다시 실행 → 이어서 학습. 6번은 저장 확인 통과 때만 런타임 반납.


In [ ]:
# 1. GPU 확인
import torch, subprocess
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (런타임 → 런타임 유형 변경)'
p = torch.cuda.get_device_properties(0)
print(p.name, f'{p.total_memory/2**30:.1f}GB', 'bf16', torch.cuda.is_bf16_supported(), 'capability', torch.cuda.get_device_capability(0))
if 'T4' in p.name: print('⚠️ T4는 bf16이 없어 느리고 메모리가 부족할 수 있습니다. A100으로 바꾸세요.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
# 2. 드라이브 연결과 파일 확인
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/ner_dapt'
for f in ['econ_dapt_morph.txt.gz', 'dapt_colab.py', 'models/dapt_mmbert_econ5m_morph/model.safetensors', 'cache/tok_econ_dapt_morph_jhu-clsp_mmBERT-base.json']:
    path = f'{ROOT}/{f}'
    print(f, os.path.exists(path), f'{os.path.getsize(path)/2**20:.0f}MB' if os.path.exists(path) else '← 없음')
assert os.path.getsize(f'{ROOT}/econ_dapt_morph.txt.gz') == 319_434_726, '말뭉치 파일이 아직 덜 올라왔습니다(드라이브 동기화 대기)'
import subprocess
assert '--train_seed' in open(f'{ROOT}/dapt_colab.py', encoding='utf-8').read(), 'dapt_colab.py 가 옛 판입니다(드라이브 동기화 대기)'
print('1에포크 가중치·캐시·새 스크립트 확인 OK')

In [ ]:
# 3. 패키지 (ModernBERT 구조는 transformers 4.48 이상 필요)
import transformers
from packaging import version
print('transformers', transformers.__version__)
if version.parse(transformers.__version__) < version.parse('4.48'):
    !pip -q install -U transformers accelerate
    print('업그레이드함 → 런타임 → 세션 다시 시작 후 1번 셀부터 다시 실행')

In [ ]:
# 4. 2번째 에포크 학습 (끊기면 1~4번을 다시 실행 → 이어서 학습)
NAME = 'dapt_mmbert_econ5m_morph_ep2'
INIT = f'{ROOT}/models/dapt_mmbert_econ5m_morph'
!cd /content && python {ROOT}/dapt_colab.py --corpus {ROOT}/econ_dapt_morph.txt.gz --out_root {ROOT} --name {NAME}     --model {INIT} --tok_from jhu-clsp/mmBERT-base --seed 42 --train_seed 43     --max_len 512 --tokens_per_step 32768 --mlm 0.15 --lr 3e-5 --epochs 1


In [ ]:
# 5. 저장 확인
import json, os
rec = json.load(open(f'{ROOT}/results/{NAME}.json', encoding='utf-8'))
for k in ['gpu', 'micro_batch', 'grad_accum', 'tokens_per_step', 'steps', 'eval_loss_before', 'eval_loss_after', 'train_minutes_this_run', 'tensors_checked', 'token_cls_shared_tensors', 'verify_ok']:
    print(f'{k:>26}: {rec.get(k)}')
print(os.listdir(f'{ROOT}/models/{NAME}'))
VERIFY_OK = bool(rec.get('verify_ok')) and os.path.exists(f'{ROOT}/models/{NAME}/model.safetensors')
print('VERIFY_OK =', VERIFY_OK)
print('비교: 1에포크 끝 0.602 →', round(rec.get('eval_loss_after', float('nan')), 4), '(같은 검증 블록)')

In [ ]:
# 6. 런타임 반납 (저장 확인이 통과했을 때만)
if VERIFY_OK:
    from google.colab import runtime
    print('가중치 저장 확인 완료 → 런타임 반납'); runtime.unassign()
else:
    print('저장 확인이 통과하지 않아 반납하지 않습니다. 5번 셀 출력과 학습 로그를 확인하세요.')